# C Code Generation

Turning a block diagram into standalone C99, and proving the C still solves the same problem.

## What comes out

`to_c` lowers the assembled model to self-contained C99 — libm and nothing else. All state lives in one instance struct, so the code is reentrant and several generated models can link into the same binary. That is what makes it usable on a microcontroller, in a HIL rig, or inside an FMU.

The generated code is not a transcription of the Python. The model first becomes the typed IR, where every block is reduced to scalar operations; the C is emitted from that.

## The Model

A DC motor under PI speed control — a small system with a controller, an integrator and feedback, which is the shape most embedded models have:

$$J\dot{\omega} = K_t i - b\omega, \qquad u = K_p e + K_i\!\int\! e\,dt, \qquad e = \omega_{\text{ref}} - \omega$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Apply the FastSim docs matplotlib style
plt.style.use('../fastsim_docs.mplstyle')

import fastsim as fs
from fastsim import Simulation, Connection
from fastsim.blocks import Source, Adder, Amplifier, Integrator, ODE, Scope
from fastsim.solvers import RK4

print(f"fastsim {fs.__version__}")

In [ ]:
J, b, K_t = 0.01, 0.02, 0.05     # inertia, friction, torque constant
K_p, K_i = 0.4, 2.5              # controller gains
OMEGA_REF = 100.0                # setpoint [rad/s]

DT, T_END = 1e-3, 2.0

def build():
    ref = Source(lambda t: OMEGA_REF)
    err = Adder("+-")
    kp = Amplifier(gain=K_p)
    ki = Amplifier(gain=K_i)
    integ = Integrator()
    u = Adder("++")
    motor = ODE(lambda x, u, t: (K_t * u[0] - b * x) / J, initial_value=0.0)
    sco = Scope(labels=["omega"], sampling_period=DT)

    sim = Simulation(
        blocks=[ref, err, kp, ki, integ, u, motor, sco],
        connections=[
            Connection(ref, err[0]),
            Connection(motor, err[1], sco),
            Connection(err, kp, integ),
            Connection(integ, ki),
            Connection(kp, u[0]),
            Connection(ki, u[1]),
            Connection(u, motor),
        ],
        Solver=RK4, dt=DT, log=False,
    )
    return sim, sco

## The Reference Run

In [ ]:
sim, sco = build()
sim.run(T_END, reset=True, adaptive=False)
t_ref, [omega_ref_traj] = sco.read()

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(t_ref, omega_ref_traj)
ax.axhline(OMEGA_REF, color="#7F7F7F", lw=1, ls=":")
ax.set_xlabel("time [s]")
ax.set_ylabel("ω [rad/s]")
plt.show()

print(f"ω({T_END}) = {omega_ref_traj[-1]:.6f} rad/s")

## Generating the C

`to_c` returns the sources keyed by file name.

In [ ]:
sim_c, _ = build()
files = sim_c.to_c("motor")

for name, src in files.items():
    print(f"{name:<12} {len(src.splitlines()):5d} lines, {len(src) / 1024:.1f} KiB")

### The interface

The header is the whole contract: one struct holding the state, an enum naming every addressable signal, and a handful of functions.

In [ ]:
header = files["motor.h"]
start = header.index("typedef struct")
print(header[start:start + 900])

In [ ]:
for line in files["motor.h"].splitlines():
    if line.startswith(("void motor_", "double motor_", "int motor_")):
        print(line)

Nothing is global, so two instances of the model can run side by side in one process — the property that makes the code safe to drop into an RTOS task or an FMU.

Signals are reached by name through the enum rather than by digging into the struct layout: `motor_get_signal(&m, MOTOR_SIG_ODE_y)` is the motor's output wherever the code generator decided to put it.

## Verification: Software in the Loop

Reading the C is not evidence. `verify_c` compiles it with a local C compiler, integrates the binary and the engine over the same trajectory, and compares the state vector at every step.

The figure it reports is a *scaled* error, $|c - \text{ref}| / (\text{atol} + \text{rtol}\,|\text{ref}|)$, so a value below 1 means the C stayed inside the tolerance at every step of every state.

In [ ]:
sim_v, _ = build()
report = sim_v.verify_c("motor", duration=T_END, dt=DT)

for key in ("passed", "compiler", "n_steps", "n_states", "max_scaled_error", "worst_state", "worst_time"):
    print(f"  {key:<18} {report[key]}")

## Compiling and Running It Ourselves

The same C, driven the way an application would: write the files out, add a `main` that steps the model, compile, and run it.

In [ ]:
import os, subprocess, tempfile
from pathlib import Path
from fastsim._fastsim import find_c_compiler

cc = find_c_compiler()
print(f"compiler: {cc}")

workdir = Path(tempfile.mkdtemp(prefix="fastsim_c_"))
for name, src in files.items():
    (workdir / name).write_text(src)

main_c = f'''
#include <stdio.h>
#include "motor.h"

int main(void) {{
    motor_t m;
    motor_init(&m);
    const double dt = {DT!r};
    const int steps = {int(round(T_END / DT))};
    for (int i = 0; i < steps; i++) {{
        motor_step(&m, dt);
        printf("%.17g %.17g\\n", m.time, motor_get_signal(&m, MOTOR_SIG_ODE_y));
    }}
    return 0;
}}
'''
(workdir / "main.c").write_text(main_c)
print("\n".join(main_c.strip().splitlines()[:6]) + " ...")

In [ ]:
import os
# gcc appends .exe on Windows; name the target so the path is right either way.
exe = workdir / ("motor_sim.exe" if os.name == "nt" else "motor_sim")
compile_cmd = [*cc.split(), "-O2", "-std=c99", "-o", str(exe),
               str(workdir / "motor.c"), str(workdir / "main.c"), "-lm"]
proc = subprocess.run(compile_cmd, capture_output=True, text=True, cwd=workdir)
print(f"compile: {'ok' if proc.returncode == 0 else proc.stderr[:400]}")

run = subprocess.run([str(exe)], capture_output=True, text=True)
rows = np.array([[float(v) for v in ln.split()] for ln in run.stdout.strip().splitlines()])
t_c, omega_c = rows[:, 0], rows[:, 1]
print(f"{len(t_c)} steps from the compiled binary, ω(end) = {omega_c[-1]:.6f} rad/s")

In [ ]:
# The driver prints after each step, so its last sample sits one step from the
# engine's. Compare at the same instant, not at the same array index — ω is still
# climbing by ~1e-3 rad/s per step here, which is what an off-by-one would show up as.
t_end_c = t_c[-1]
engine_here = np.interp(t_end_c, t_ref, omega_ref_traj)
print(f"at t = {t_end_c:.6f} s")
print(f"  engine : {engine_here:.9f} rad/s")
print(f"  C      : {omega_c[-1]:.9f} rad/s")
print(f"  differ : {abs(omega_c[-1] - engine_here):.3e} rad/s")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t_ref, omega_ref_traj, label="FastSim engine")
ax.plot(t_c, omega_c, "--", label="generated C")
ax.axhline(OMEGA_REF, color="#7F7F7F", lw=1, ls=":")
ax.set_xlabel("time [s]")
ax.set_ylabel("ω [rad/s]")
ax.legend()
plt.show()

## Code Generation Options

The same model lowers several ways. The axes are independent, and every combination is checked by the test suite against the engine.

In [ ]:
variants = [
    ("double, hierarchical", dict(numeric="double", structure="hierarchical")),
    ("double, flat",         dict(numeric="double", structure="flat")),
    ("float,  hierarchical", dict(numeric="float",  structure="hierarchical")),
    ("double, vectorized",   dict(numeric="double", reductions="vectorized")),
]

for label, opts in variants:
    sim_o, _ = build()
    out = sim_o.to_c("motor", **opts)
    total = sum(len(s.splitlines()) for s in out.values())
    print(f"  {label:<22} {total:5d} lines")

`numeric="float"` is the one that changes the answer rather than just the shape: single precision costs about seven digits, which matters on a target without a double-precision FPU. The verification above is what tells you whether that is affordable for a given model.